# Houston Hurricane Response Planner

This notebook runs the whole project in order: setup, numerical model training and validation, then an interactive staging and cooling-hub planner.

1. Download the companion **houston_hurricane_hackathon.zip** to your computer.
2. In Colab's key-icon **Secrets** panel, save **CARTO_API_KEY** and enable **Notebook access**. An existing saved key can be reused.
3. Run the cells in order, or use **Run all**. When the first cell shows **Choose Files**, select the project ZIP.

This is a historical Harvey planning case study. It does not predict future power outages or guarantee rescue times. No GPU is needed.


## 1 Load the project
The setup cell uploads and unpacks the ZIP and installs the Python libraries. After the project is unpacked, rerunning this cell in the same runtime skips the upload. Colab runtime files are temporary.


In [ ]:
from pathlib import Path
import io
import zipfile
from google.colab import files

project = Path("/content/houston_hurricane_project")
if not (project / "train_damage_model.py").exists():
    print("Choose the downloaded houston_hurricane_hackathon.zip file.")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError("Please upload exactly one project ZIP file.")
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_names[0]])) as archive:
        root = Path("/content").resolve()
        for member in archive.infolist():
            target = (root / member.filename).resolve()
            if not target.is_relative_to(project.resolve()):
                raise ValueError("The ZIP contains an unexpected project path.")
        archive.extractall(root)

if not (project / "complete_planner_with_hubs.py").exists():
    raise FileNotFoundError("The project ZIP did not contain the planner.")
print("Project files are ready.")
%pip -q install -r /content/houston_hurricane_project/requirements.txt


## 2 Train and evaluate the model

The model estimates the number of registered homeowner cases with recorded inspected damage over $30,000. Inputs are already-reported registration counts and location. The damage threshold is a project definition, not an official structural damage classification.

The code holds out 25% of unique study ZIP areas, trains a random forest on the remaining ZIPs, and compares its mean absolute error (MAE) against a baseline using a pooled high-damage fraction. Lower MAE is better. Planning estimates are generated with three-fold cross-fitting, leaving each ZIP's target out of the fit predicting that ZIP.

All training and evaluation are within Harvey. This is post-storm estimation, not validation on a future hurricane. No damage-amount or payment fields are model inputs.


In [ ]:
%run /content/houston_hurricane_project/train_damage_model.py


In [ ]:
import json
import pandas as pd
from pathlib import Path

report = json.loads(Path("/content/houston_hurricane_project/model_report.json").read_text())
display(pd.DataFrame([
    {"Method": "Pooled-rate baseline", "MAE: high-damage cases per ZIP": report["baseline_mae_cases_per_zip"]},
    {"Method": "Random forest", "MAE: high-damage cases per ZIP": report["model_mae_cases_per_zip"]}
]).round(2))
print("Held-out ZIP areas:", report["test_zip_codes"])
print(f"MAE reduction versus baseline: {report['mae_reduction_percent']:.2f}%")


## 3 Run the interface

The **Priority** menu switches between observed registration weight and model-estimated high-damage case weight. Percentages for different priority metrics are not directly comparable.

Use **Bases** to change the staging-site budget, **Slowdown** for an assumed travel multiplier, and **Exclude** to simulate one unavailable site. The optimizer evaluates every eligible site combination, maximizing covered priority weight within a modeled ten-minute base-to-area drive, then breaking ties by weighted average driving time.

The example FEMA screen excludes mapped Special Flood Hazard Area points and unknown results. Passing this screen does not establish facility safety. Eligible community centers are only assumed-open cooling hubs; activation and capacity are unverified. The two-leg driving table excludes rescue/loading time, queues, return to base and hospital transport.


In [ ]:
%run /content/houston_hurricane_project/complete_planner_with_hubs.py


## 4 Present and submit

Use **Historical registrations**, **2 bases**, **1.0 slowdown**, and **None excluded** for the reproducible placement example. Expected coverage is about **41.54% of registration weight in this ten-point sample**. The five-candidate baseline covered about **23.98%** using the same two-site budget.

Switch to model-estimated priority to explain the trained component. Exclude Northeast to demonstrate replanning. Read the presentation notes below and use the actual model metrics printed by your run.

Prepare an accessible code/notebook link and a live demo. The supplied Kaggle description requests submission by 4:30 PM Central; the packet also asks for a live demo. Confirm any day-of organizer updates.


In [ ]:
from IPython.display import Markdown, display
display(Markdown(Path("/content/houston_hurricane_project/PRESENTATION.md").read_text()))
